# FIT5226 Stage 2 — Multi-Agent Reinforcement Learning

## Grid Layout
```
     col0  col1  col2  col3  col4
row0  [  ]  [  ]  [ Y]  [  ]  [  ]
row1  [  ]  [  ]  [  ]  [  ]  [  ]
row2  [ X]  [  ]  [LK]  [  ]  [ U]
row3  [  ]  [  ]  [  ]  [  ]  [  ]
row4  [  ]  [  ]  [ V]  [  ]  [  ]
```
- **X** = (2,0): Type A start/delivery  
- **Y** = (0,2): Type B start/delivery  
- **U** = (2,4): Type A sample site  
- **V** = (4,2): Type B sample site  
- **LAKE** = (2,2): shared crossing, probabilistically floods/dries

Shortest path A: X→(2,1)→LAKE→(2,3)→U and back.  
Shortest path B: Y→(1,2)→LAKE→(3,2)→V and back.  
Both paths intersect only at LAKE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import process_time

# Fixed grid locations
X    = (2, 0)
Y    = (0, 2)
U    = (2, 4)
V    = (4, 2)
LAKE = (2, 2)

GRID_ROWS    = 5
GRID_COLS    = 5
NUM_ACTIONS  = 5   # 0=Up 1=Down 2=Left 3=Right 4=Wait
LAKE_FLIP_PROB = 0.3  # probability lake flips state each timestep

# Rewards
R_STEP      = -5
R_WAIT      = -3
R_COLLISION = -20
R_WATER_DMG = -20   # Phase 1 only: Type A enters flooded lake
R_PICKUP    = 10
R_DELIVER   = 50

In [ ]:
def movingAverage(arr, window_size):
    return [round(np.sum(arr[i:i+window_size]) / window_size, 2)
            for i in range(len(arr) - window_size + 1)]


def plot_results(rewards_a, rewards_b, collisions_log, success_log, title, window=500):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(title, fontsize=14)

    axes[0, 0].plot(movingAverage(rewards_a, window))
    axes[0, 0].set(title='Agent A — Reward', xlabel='Episode', ylabel='Avg Reward')

    axes[0, 1].plot(movingAverage(rewards_b, window), color='orange')
    axes[0, 1].set(title='Agent B — Reward', xlabel='Episode', ylabel='Avg Reward')

    axes[1, 0].plot(movingAverage(collisions_log, window), color='red')
    axes[1, 0].set(title='Collision Rate', xlabel='Episode', ylabel='Collisions / Episode')

    axes[1, 1].plot(movingAverage(success_log, window), color='green')
    axes[1, 1].set(title='Success Rate (both delivered)', xlabel='Episode', ylabel='Rate')

    plt.tight_layout()
    plt.show()


def visualise_policy(agent, agent_name):
    """Print action chosen at the approach cell just before the lake."""
    approach = {'A': (2, 1), 'B': (1, 2)}
    cell = approach[agent_name]
    action_names = ['Up', 'Down', 'Left', 'Right', 'Wait']
    print(f"\nPolicy for Agent {agent_name} at approach cell {cell}:")
    for carrying in [0, 1]:
        label = 'going to sample' if carrying == 0 else 'returning'
        for lake in [0, 1]:
            state = (cell[0], cell[1], carrying, lake)
            best = np.argmax(agent.q_table[state])
            q_val = agent.q_table[state + (best,)]
            print(f"  {label}, lake={'flooded' if lake else 'dry':>7}: "
                  f"{action_names[best]:<5}  (Q={q_val:.2f})")

In [ ]:
class MultiAgentEnvironment:
    """Owns all world state for both agents and the lake."""

    def __init__(self):
        self.lake_state = 0  # 0=dry, 1=flooded
        self._reset()

    def _reset(self):
        self.pos_a      = X
        self.pos_b      = Y
        self.carrying_a = 0  # 0=heading to U, 1=returning to X
        self.carrying_b = 0  # 0=heading to V, 1=returning to Y
        self.done_a     = False
        self.done_b     = False

    def get_state_a(self):
        return (self.pos_a[0], self.pos_a[1], self.carrying_a, self.lake_state)

    def get_state_b(self):
        return (self.pos_b[0], self.pos_b[1], self.carrying_b, self.lake_state)

    @staticmethod
    def _apply_action(pos, action):
        r, c = pos
        if   action == 0: r = max(r - 1, 0)
        elif action == 1: r = min(r + 1, GRID_ROWS - 1)
        elif action == 2: c = max(c - 1, 0)
        elif action == 3: c = min(c + 1, GRID_COLS - 1)
        # action == 4 (Wait): position unchanged
        return (r, c)

    def step(self, action_a, action_b, phase=1):
        """
        Execute one simultaneous timestep.
        Returns (reward_a, reward_b, collision_occurred).
        Sequence follows spec: move → collision+rewards → lake flip.
        """
        # Step 3: execute actions
        next_a = self._apply_action(self.pos_a, action_a)
        next_b = self._apply_action(self.pos_b, action_b)

        # Step 4+5: rewards
        reward_a = R_WAIT if action_a == 4 else R_STEP
        reward_b = R_WAIT if action_b == 4 else R_STEP
        collision = False

        # Collision: both land on the lake cell
        if next_a == LAKE and next_b == LAKE:
            reward_a += R_COLLISION
            reward_b += R_COLLISION
            collision = True

        # Phase 1 only: Type A penalised for entering flooded lake
        if phase == 1 and next_a == LAKE and self.lake_state == 1:
            reward_a += R_WATER_DMG

        # Commit moves
        self.pos_a = next_a
        self.pos_b = next_b

        # Sample pickup (step into site while not yet carrying)
        if self.pos_a == U and self.carrying_a == 0:
            self.carrying_a = 1
            reward_a += R_PICKUP
        if self.pos_b == V and self.carrying_b == 0:
            self.carrying_b = 1
            reward_b += R_PICKUP

        # Sample delivery (return to ship with sample)
        if self.pos_a == X and self.carrying_a == 1:
            self.carrying_a = 0
            reward_a += R_DELIVER
            self.done_a = True
        if self.pos_b == Y and self.carrying_b == 1:
            self.carrying_b = 0
            reward_b += R_DELIVER
            self.done_b = True

        # Step 6: lake flips probabilistically
        if np.random.random() < LAKE_FLIP_PROB:
            self.lake_state = 1 - self.lake_state

        return reward_a, reward_b, collision

In [ ]:
class QTableAgent2:
    """
    Tabular Q-learning agent for the Stage 2 grid world.
    Q-table shape: (5, 5, 2, 2, 5) = row x col x carrying x lake_state x action.
    """

    def __init__(self):
        self.q_table = np.zeros((GRID_ROWS, GRID_COLS, 2, 2, NUM_ACTIONS))

    def choose_action(self, state, epsilon):
        if np.random.random() < epsilon:
            return np.random.randint(NUM_ACTIONS)
        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, next_pos_carrying, reward, lr, gamma, done):
        """
        Q-learning update accounting for both possible next lake states.
        next_pos_carrying = (row, col, carrying) — lake state excluded
        because it may flip before the next observation.
        Per spec Hint 4: expected future value averages over lake flip probability.
        """
        row, col, carrying = next_pos_carrying
        current_lake = state[3]

        if done:
            max_future_q = 0.0
        else:
            q_same   = np.max(self.q_table[row, col, carrying, current_lake])
            q_flip   = np.max(self.q_table[row, col, carrying, 1 - current_lake])
            max_future_q = (1 - LAKE_FLIP_PROB) * q_same + LAKE_FLIP_PROB * q_flip

        td_target = reward + gamma * max_future_q
        td_error  = td_target - self.q_table[state + (action,)]
        self.q_table[state + (action,)] += lr * td_error

In [ ]:
def train(agent_a, agent_b, env,
          num_episodes=100_000, max_steps=200,
          lr=0.3, gamma=0.95,
          eps_init=1.0, eps_final=0.05, eps_decay=0.99997,
          phase=1):
    """
    Joint training loop for both agents.
    Episode ends when both agents complete their trip or max_steps is reached.
    Returns per-episode lists: rewards_a, rewards_b, collisions, successes.
    """
    epsilon   = eps_init
    rewards_a = []
    rewards_b = []
    collisions_log = []
    success_log    = []

    for episode in range(num_episodes):
        env._reset()
        ep_r_a = ep_r_b = ep_col = 0
        step = 0

        while step < max_steps and not (env.done_a and env.done_b):
            state_a = env.get_state_a()
            state_b = env.get_state_b()

            # Done agents wait in place
            act_a = 4 if env.done_a else agent_a.choose_action(state_a, epsilon)
            act_b = 4 if env.done_b else agent_b.choose_action(state_b, epsilon)

            r_a, r_b, collision = env.step(act_a, act_b, phase=phase)

            next_state_a = env.get_state_a()
            next_state_b = env.get_state_b()

            if not env.done_a or step == 0:  # update until just-done
                agent_a.update(
                    state_a, act_a,
                    (next_state_a[0], next_state_a[1], next_state_a[2]),
                    r_a, lr, gamma, env.done_a
                )
            if not env.done_b or step == 0:
                agent_b.update(
                    state_b, act_b,
                    (next_state_b[0], next_state_b[1], next_state_b[2]),
                    r_b, lr, gamma, env.done_b
                )

            ep_r_a += r_a
            ep_r_b += r_b
            if collision:
                ep_col += 1
            step += 1

        epsilon = max(eps_final, epsilon * eps_decay)
        rewards_a.append(ep_r_a)
        rewards_b.append(ep_r_b)
        collisions_log.append(ep_col)
        success_log.append(1 if (env.done_a and env.done_b) else 0)

        if episode % 10_000 == 0:
            print(f"Episode {episode:>7}  ε={epsilon:.4f}  "
                  f"avg_r_a={np.mean(rewards_a[-1000:]):.1f}  "
                  f"avg_r_b={np.mean(rewards_b[-1000:]):.1f}  "
                  f"col_rate={np.mean(collisions_log[-1000:]):.3f}")

    return rewards_a, rewards_b, collisions_log, success_log

---
## Task 1 — Phase 1: Asymmetric Waterproofing

Type A has unreliable waterproofing: penalty of **-20** for entering the lake when flooded.  
Type B is fully waterproofed (no water penalty).  
Collision penalty applies to both types regardless.

**Expected outcome:** A learns to cross only when the lake is **dry**; B consequently learns to cross only when the lake is **flooded** (no A present → collision-free).

In [ ]:
t0 = process_time()

np.random.seed(42)
env1    = MultiAgentEnvironment()
agent_a1 = QTableAgent2()
agent_b1 = QTableAgent2()

ra1, rb1, col1, suc1 = train(
    agent_a1, agent_b1, env1,
    num_episodes=100_000,
    max_steps=200,
    lr=0.3, gamma=0.95,
    eps_init=1.0, eps_final=0.05, eps_decay=0.99997,
    phase=1
)

print(f"\nPhase 1 training completed in {process_time()-t0:.1f}s")
print(f"Final 5000-ep collision rate : {np.mean(col1[-5000:]):.4f}")
print(f"Final 5000-ep success rate   : {np.mean(suc1[-5000:]):.4f}")

In [ ]:
plot_results(ra1, rb1, col1, suc1, title='Phase 1 — Asymmetric Waterproofing', window=500)

In [ ]:
visualise_policy(agent_a1, 'A')
visualise_policy(agent_b1, 'B')

In [ ]:
agent_a1.q_table.astype('float32').tofile('qTable_A_phase1.dat')
agent_b1.q_table.astype('float32').tofile('qTable_B_phase1.dat')
print('Phase 1 Q-tables saved.')

---
## Task 2 — Phase 2: Full Waterproofing (Robert's Decree)

Both robot types now have full waterproofing. There is **no penalty** for Type A entering the flooded lake.  
Collision danger at the lake remains. Agents must learn traffic-light behaviour (one crosses when dry, the other when flooded) purely through Q-learning — without any asymmetric incentive to break the symmetry.

**Sarah's prediction:** This will be extremely difficult. Without an external gradient distinguishing the two equilibria, Q-learning agents have no mechanism to settle on one convention.

In [ ]:
t0 = process_time()

np.random.seed(42)
env2     = MultiAgentEnvironment()
agent_a2 = QTableAgent2()
agent_b2 = QTableAgent2()

ra2, rb2, col2, suc2 = train(
    agent_a2, agent_b2, env2,
    num_episodes=100_000,
    max_steps=200,
    lr=0.3, gamma=0.95,
    eps_init=1.0, eps_final=0.05, eps_decay=0.99997,
    phase=2
)

print(f"\nPhase 2 training completed in {process_time()-t0:.1f}s")
print(f"Final 5000-ep collision rate : {np.mean(col2[-5000:]):.4f}")
print(f"Final 5000-ep success rate   : {np.mean(suc2[-5000:]):.4f}")

In [ ]:
plot_results(ra2, rb2, col2, suc2, title='Phase 2 — Full Waterproofing', window=500)

In [ ]:
# Direct comparison of collision rates
w = 500
plt.figure(figsize=(10, 4))
plt.plot(movingAverage(col1, w), label='Phase 1 (asymmetric)', color='blue')
plt.plot(movingAverage(col2, w), label='Phase 2 (symmetric)',  color='red', alpha=0.8)
plt.xlabel('Episode')
plt.ylabel('Avg Collisions / Episode')
plt.title('Collision Rate: Phase 1 vs Phase 2')
plt.legend()
plt.tight_layout()
plt.show()

visualise_policy(agent_a2, 'A')
visualise_policy(agent_b2, 'B')

In [ ]:
agent_a2.q_table.astype('float32').tofile('qTable_A_phase2.dat')
agent_b2.q_table.astype('float32').tofile('qTable_B_phase2.dat')
print('Phase 2 Q-tables saved.')

### Phase 1 vs Phase 2 — Analysis

**Phase 1** converges to near-zero collision rate because Type A's water-damage penalty creates an independent, individual incentive: crossing a flooded lake is always costly for A regardless of what B does. A therefore learns to avoid the flooded lake unilaterally. Once A reliably stays out of the lake when flooded, B can cross safely during floods with no collision risk. The asymmetric penalty selects a unique equilibrium.

**Phase 2** struggles because the symmetry is restored. Both agents face identical payoff structures at the lake: crossing is fine as long as the other agent does not cross simultaneously. There are two equally valid pure-strategy equilibria (A crosses when dry / B crosses when flooded, or vice versa), but Q-learning provides no gradient to distinguish them. Each agent's Q-values evolve in response to the other's behaviour, and without a tiebreaker the agents may oscillate or settle on a mixed strategy with persistent collisions.

### Distinction Experiment — Effect of Step Penalty on Phase 2 Coordination

We vary the step penalty while keeping the collision penalty fixed at -20.  
**Hypothesis:** A higher step penalty makes waiting relatively cheaper compared to the collision cost, which may nudge agents toward more cautious (wait-based) policies and accidentally improve coordination.

In [ ]:
step_penalties = [-1, -5, -10, -20]
exp_collision_rates = []
exp_success_rates   = []

for sp in step_penalties:
    # Temporarily override R_STEP
    R_STEP_orig = R_STEP
    import builtins
    # Use a subclassed env that overrides step reward
    class ExpEnv(MultiAgentEnvironment):
        def step(self, action_a, action_b, phase=2):
            next_a = self._apply_action(self.pos_a, action_a)
            next_b = self._apply_action(self.pos_b, action_b)
            reward_a = R_WAIT if action_a == 4 else sp
            reward_b = R_WAIT if action_b == 4 else sp
            collision = False
            if next_a == LAKE and next_b == LAKE:
                reward_a += R_COLLISION
                reward_b += R_COLLISION
                collision = True
            self.pos_a = next_a
            self.pos_b = next_b
            if self.pos_a == U and self.carrying_a == 0:
                self.carrying_a = 1; reward_a += R_PICKUP
            if self.pos_b == V and self.carrying_b == 0:
                self.carrying_b = 1; reward_b += R_PICKUP
            if self.pos_a == X and self.carrying_a == 1:
                self.carrying_a = 0; reward_a += R_DELIVER; self.done_a = True
            if self.pos_b == Y and self.carrying_b == 1:
                self.carrying_b = 0; reward_b += R_DELIVER; self.done_b = True
            if np.random.random() < LAKE_FLIP_PROB:
                self.lake_state = 1 - self.lake_state
            return reward_a, reward_b, collision

    np.random.seed(42)
    exp_env = ExpEnv()
    exp_a   = QTableAgent2()
    exp_b   = QTableAgent2()
    _, _, c, s = train(exp_a, exp_b, exp_env,
                       num_episodes=50_000, max_steps=200,
                       lr=0.3, gamma=0.95,
                       eps_init=1.0, eps_final=0.05, eps_decay=0.99997,
                       phase=2)
    exp_collision_rates.append(np.mean(c[-5000:]))
    exp_success_rates.append(np.mean(s[-5000:]))
    print(f"step_penalty={sp:>4}: collision_rate={exp_collision_rates[-1]:.4f}  "
          f"success_rate={exp_success_rates[-1]:.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar([str(p) for p in step_penalties], exp_collision_rates, color='red', alpha=0.7)
ax1.set(title='Phase 2: Collision Rate vs Step Penalty',
        xlabel='Step Penalty', ylabel='Avg Collisions/Episode (last 5000)')
ax2.bar([str(p) for p in step_penalties], exp_success_rates, color='green', alpha=0.7)
ax2.set(title='Phase 2: Success Rate vs Step Penalty',
        xlabel='Step Penalty', ylabel='Success Rate (last 5000)')
plt.tight_layout()
plt.show()

---
## Task 3 — Game-Theoretic Analysis with Replicator Dynamics

We reduce the scenario to the core crossing decision, assuming agents already know how to navigate the shortest path. Each agent's only decision is: **Cross** (C) or **Wait** (W) at the lake cell.

We define payoffs for a single crossing encounter and apply replicator dynamics to find equilibrium behaviour.

In [ ]:
# Payoff matrices — rows = Agent A strategy, cols = Agent B strategy
# Strategies: 0=Cross, 1=Wait
# Payoffs represent the net gain at the crossing decision point.
# Cross successfully: 0 extra cost.  Wait: -3 (wait penalty vs -5 step = +2 saved, but
# we normalise the base step cost out and focus on differential payoffs).
# Collision cost: -20 for both.
# Phase 1: A additionally pays -20 for crossing flooded lake;
#          but since A only crosses when dry, we focus on dry-lake decisions here.

# Differential payoff (relative to baseline step cost):
#   Cross vs Cross: collision => -20
#   Cross vs Wait:  safe cross => 0 (A crosses), Wait saves 2 for B
#   Wait vs Cross:  Wait saves 2 for A, B crosses safely => 0
#   Wait vs Wait:   both wait => +2 each (saved collision risk, both pay wait not step)

# Phase 2 payoff matrix (symmetric)
# Row player = A, Col player = B
# Values: (payoff_A, payoff_B)
payoff_A_p2 = np.array([
    [-20,   0],   # A=Cross: [B=Cross, B=Wait]
    [ -3,  -3],   # A=Wait:  [B=Cross, B=Wait]
], dtype=float)

payoff_B_p2 = np.array([
    [-20,  -3],   # A=Cross: [B=Cross, B=Wait]
    [  0,  -3],   # A=Wait:  [B=Cross, B=Wait]
], dtype=float)

# Phase 1 payoff matrix (asymmetric: A penalised for crossing when flooded)
# In dry-lake context, Phase 1 = Phase 2 (no water penalty when dry).
# In flooded-lake context, A additionally pays -20 for crossing:
payoff_A_p1_flooded = np.array([
    [-40,  -20],  # A=Cross(flooded): collision adds -20 on top of water -20
    [ -3,   -3],
], dtype=float)

payoff_B_p1_flooded = np.array([
    [-20,   -3],
    [  0,   -3],
], dtype=float)

print("Phase 2 payoff matrix (A):")
print(payoff_A_p2)
print("\nPhase 2 payoff matrix (B):")
print(payoff_B_p2)
print("\nPhase 1 payoff matrix A (flooded lake):")
print(payoff_A_p1_flooded)

In [ ]:
def replicator_dynamics(payoff_A, payoff_B, p0_A, p0_B, steps=20_000, dt=0.005):
    """
    Simulate replicator dynamics for a 2x2 asymmetric game.
    p_A = probability A plays Cross (strategy 0).
    p_B = probability B plays Cross (strategy 0).
    Returns trajectory array of shape (steps+1, 2).
    """
    p_A, p_B = p0_A, p0_B
    history = [(p_A, p_B)]

    for _ in range(steps):
        # Expected payoffs for each strategy
        f_AC = p_B * payoff_A[0, 0] + (1 - p_B) * payoff_A[0, 1]  # A plays Cross
        f_AW = p_B * payoff_A[1, 0] + (1 - p_B) * payoff_A[1, 1]  # A plays Wait
        f_avg_A = p_A * f_AC + (1 - p_A) * f_AW

        f_BC = p_A * payoff_B[0, 0] + (1 - p_A) * payoff_B[1, 0]  # B plays Cross
        f_BW = p_A * payoff_B[0, 1] + (1 - p_A) * payoff_B[1, 1]  # B plays Wait
        f_avg_B = p_B * f_BC + (1 - p_B) * f_BW

        # Replicator equation: dp/dt = p * (f_strategy - f_avg)
        p_A = np.clip(p_A + dt * p_A * (f_AC - f_avg_A), 0, 1)
        p_B = np.clip(p_B + dt * p_B * (f_BC - f_avg_B), 0, 1)
        history.append((p_A, p_B))

    return np.array(history)


# Run from a grid of initial conditions
init_conditions = [(p_a, p_b)
                   for p_a in np.linspace(0.05, 0.95, 6)
                   for p_b in np.linspace(0.05, 0.95, 6)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (pA_mat, pB_mat, title) in zip(axes, [
    (payoff_A_p1_flooded, payoff_B_p1_flooded, 'Phase 1 (flooded lake, asymmetric)'),
    (payoff_A_p2,         payoff_B_p2,         'Phase 2 (symmetric)'),
]):
    for (p0a, p0b) in init_conditions:
        traj = replicator_dynamics(pA_mat, pB_mat, p0a, p0b)
        ax.plot(traj[:, 0], traj[:, 1], alpha=0.4, linewidth=0.8, color='steelblue')
        ax.plot(traj[0, 0],  traj[0, 1],  'o', markersize=3, color='green')
        ax.plot(traj[-1, 0], traj[-1, 1], 's', markersize=3, color='red')

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('p(A crosses)')
    ax.set_ylabel('p(B crosses)')
    ax.set_title(title)
    ax.axhline(0.5, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(0.5, color='gray', linewidth=0.5, linestyle='--')
    # Mark Nash equilibria
    ax.plot(1, 0, '*', markersize=12, color='orange', label='NE: A crosses, B waits')
    ax.plot(0, 1, '*', markersize=12, color='purple', label='NE: A waits, B crosses')
    ax.legend(fontsize=8)

plt.suptitle('Replicator Dynamics Phase Portraits\n(green=start, red=end)', fontsize=12)
plt.tight_layout()
plt.show()

### Game-Theoretic Analysis

The crossing scenario is an **anti-coordination game** (structurally similar to Hawk-Dove / Chicken). There are two pure-strategy Nash equilibria: *(A crosses, B waits)* and *(A waits, B crosses)*, and one mixed-strategy NE.

**Phase 1** (flooded lake, asymmetric): Type A's water penalty makes crossing a dominated strategy for A when the lake is flooded, regardless of B's action. The payoff matrix is asymmetric: A's dominant strategy is to Wait when flooded. Replicator dynamics converge to the unique pure equilibrium (A waits, B crosses) from virtually all initial conditions — a single basin of attraction.

**Phase 2** (symmetric): The payoff matrix is symmetric. Both pure NE are equally valid. Replicator dynamics show two basins of attraction, separated by a boundary near the mixed-strategy equilibrium. Initial conditions determine which equilibrium is reached. This is why independent Q-learning agents struggle: there is no shared mechanism to select one convention, and small fluctuations in early learning can push the system into either basin — or trap it near the unstable mixed-strategy equilibrium.

---
## Task 4 — Theoretical Explanation

Phase 1 succeeds because Type A's water-damage penalty breaks the payoff symmetry: crossing a flooded lake is individually costly for A *regardless of B's action*, giving A a dominant strategy (avoid flooded lake). This asymmetric incentive selects a unique Nash equilibrium without requiring coordination. Q-learning finds it reliably because the reward landscape has a single clear gradient.

Phase 2 fails because removing the penalty restores symmetry. The game now has two equivalent pure Nash equilibria (A crosses when dry / B waits when dry, or vice versa). Independent Q-learning agents cannot coordinate on one equilibrium: each agent's Q-values depend on the other's evolving policy, but neither can observe the other's state. The co-evolving Q-tables create a non-stationary environment for each agent, violating the stationarity assumption Q-learning requires. Replicator dynamics confirm that Phase 2 has two equal-sized basins of attraction divided by an unstable boundary — trajectory outcome is dominated by initial conditions, not a shared convention. Sarah is right.

---
## Generative AI Declaration

Claude Code (claude-sonnet-4-6) was used to assist with the implementation of this notebook. It was used for: structuring the notebook cells, writing the `MultiAgentEnvironment` and `QTableAgent2` classes, the training loop, visualisation functions, replicator dynamics simulation, and the written analysis sections.

All code has been reviewed and is fully understood by the author. The AI was treated as a non-authoritative external collaborator — all design decisions, reward structures, state representations, and theoretical interpretations were verified against the assignment specification and relevant course material.